# viel — real GPU experiment
This notebook runs the experimental encoder optimizer. It is not verified protection against Meta AI or any editor. Select **Runtime → Change runtime type → GPU**. No paid resources are provisioned by these cells. Read the model card/license before downloading weights.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/viel')
if not repo.exists():
    subprocess.run(['git','clone','--branch','setup/viel-foundation','https://github.com/indujavg/viel.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run(['git','rev-parse','HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','backend/requirements.txt'],check=True)
if str(repo) not in sys.path: sys.path.insert(0,str(repo))
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print(torch.cuda.get_device_name())


Review https://huggingface.co/stabilityai/sd-vae-ft-mse and its license. Set the flag below only after review. The resolved model revision is recorded for repeatability.


In [ ]:
MODEL_TERMS_REVIEWED = False
assert MODEL_TERMS_REVIEWED, 'Review model terms, then set MODEL_TERMS_REVIEWED = True.'
from huggingface_hub import model_info
os.environ['VIEL_MODEL_ID'] = 'stabilityai/sd-vae-ft-mse'
os.environ['VIEL_MODEL_REVISION'] = model_info(os.environ['VIEL_MODEL_ID']).sha
print('Pinned model revision:', os.environ['VIEL_MODEL_REVISION'])
from backend.engine import ShieldEngine
engine = ShieldEngine()


Upload one image you have permission to process. Colab receives it. Use an opaque PNG/JPEG with sides 32–1024 pixels; start small. This notebook creates output files in its temporary runtime.


In [ ]:
from google.colab import files
from backend.images import decode_image
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one image.'
image = decode_image(next(iter(uploaded.values())))
display(image)


In [ ]:
import threading, json
from PIL import Image
from io import BytesIO
result, report = engine.protect(image, 'subtle', 42, threading.Event())
pathlib.Path('/content/viel-candidate.png').write_bytes(result)
pathlib.Path('/content/viel-experiment.json').write_text(json.dumps(report,indent=2))
display(Image.open(BytesIO(result)))
print(json.dumps(report,indent=2))
files.download('/content/viel-candidate.png')
files.download('/content/viel-experiment.json')


## Evaluate, do not assume
Compare original/candidate visually, then perform matched AI edits of both images and repeat the experiment. See `docs/evaluation.md`. A lower encoder objective is not proof of editing disruption. This notebook does not deploy an API or automate Meta AI.
